BASE MODEL


In [ ]:
from model_system import run_hybrid_summary

results = run_hybrid_summary(desired_array_size=120000, 
                             P_ref=110, 
                             t_TES_hours=14
                             )
print(f"Capacity Factor : {results.get('CF_hybrid')*100:.2f} %")
print(f"PPA Price MUSD  : {results.get('PPA_USD_per_MWh'):.2f}")

BASE MODEL DETAILS

In [ ]:
from models.pv_model import run_pvsam
from models.lcoe import calculate_lcoe_constant
import numpy as np
from capex_opex.excel_capex_opex import calc_capex_opex
from pathlib import Path
from models.csp_model import run_basic_mspt
import sys
sys.path.append(".")
from models.dispatch_opt import (
    optimize_dispatch,
    export_dispatch_to_csv,
    calculate_firm_bonus,
    plot_week_dispatch,
    plot_stack_flows,
    plot_shares_price
)


excel_file = Path("capex_opex/CAPEX OPEX model CSP-PV MJ2500.xlsx")
# == PV ===
desired_array_size = 102000     # kWdc
dc_to_ac_ratio=1.2,             # DC to AC ratio for PV system
# == CSP ===
P_ref = 100                     # CSP plant size in MWe
solarm = 3                      # Solar Multiple
eta_PB=0.40                     # PB efficiency
# == TES ===
t_TES_hours = 20                # hours of thermal energy storage capacity
max_to_grid = 100               # MWe limit
# == ECONOMIC MODEL ===
x_price = 1                     
debt = 0.8
equity = 0.2
debt_rate = 0.05
equity_rate = 0.08
tax_rate = 0.25
lifetime_years = 30
construction_years = 2
# == MISC ===
dist_to_grid = 0                # km
dist_to_road = 0                # km
dist_to_gas = 0                 # km
dist_to_water = 0               # km
other_comp = 0                  # dummy variable for other component size

df_pv, pv_summary, pv_obj, annual_energy_kwh, land_area_m2, name_plate_kwdc = run_pvsam(
    system_kwargs=dict(
        desired_array_size=desired_array_size,  
        dc_to_ac_ratio=dc_to_ac_ratio,
    )
)

df_csp, csp_summary, mspt = run_basic_mspt(
    P_ref=P_ref,
    solarm=solarm,
)

q_csp = df_csp['P_rec_MWt'].tolist()  
e_pv  = (df_pv['AC_kWh'] / 1000).tolist() 

res, model, E_cap, PV_to_heater_max,CF_hybrid,CF_pb, E_hybrid,E_pv_grid,E_pb_elec = optimize_dispatch(
    q_csp,e_pv,eta_PB=eta_PB,x_price=x_price,max_to_grid=max_to_grid,allow_spill=True,     
    objective_mode="revenue",                                   # <---: "revenue" or "cf_hybrid" or "cf_pb" (optional)                       
    PB_e_max=csp_summary.get("NetCapacity_MWe"),
    E_cap=(csp_summary.get("NetCapacity_MWe")*t_TES_hours)/eta_PB,
    solarm=solarm,
    )  

bonus = calculate_firm_bonus(res, base_price=50.0, bonus_rate=0.2, margin=0.05, min_hours=3)

inputs = {
    "PB Installed Capacity (Gross)":            csp_summary.get("NetCapacity_MWe")*1000,                # kW
    "Solar Field Aperture Area (Mirror Area)":  csp_summary.get("Solar_Field_Area_m2"),                 # m2
    "Thermal Energy Storage Capacity ":         E_cap,
    "Receiver Power (Max Rated)":               csp_summary.get("Receiver_Design_MWt"),                 # MW 0 if Parabolic Trough
    "Tower Height (w/o Receiver)":              csp_summary.get("Tower_Height_m"),                      # m
    # "Electric Heater Thermal Power (Max Rated)":csp_summary.get("Electric_Heater_Power_MWt"),         # MWt
    "Electric Heater Thermal Power (Max Rated)":PV_to_heater_max*1000,
    "Land Area CSP":                            csp_summary.get("Land_Area_acre") * 4046.86,            # m2
    "PV Installed Capacity":                    name_plate_kwdc *1000,                                  # Wdc 
    "Battery Pack Power (Max Rated)":           0,                                                    # MW bess_energy_mwh or 
    "Battery Pack Capacity":                    0,                                                      # MWh-e bess_energy_mwh or 
    "Battery Annual Generation (for OPEX)":     0,                                                    # GWh/y
    "Land Area PV":                             land_area_m2,                                           # m2
    "CSP Annual Generation (for OPEX)":         csp_summary.get("Annual_kWh") / 1000000,                # GWh/y
    "Distance to Grid ":                        dist_to_grid,
    "Distance to Road":                         dist_to_road,
    "Distance to Gas":                          dist_to_gas,
    "Distance to Water":                        dist_to_water,
    "Other Component Size":                     other_comp,
}

overrides = {
    # If tower technology: SET BOP Reference to 74.8 MUSD or If PT technology: SET BOP Reference Cost to 90.2 MUSD
    "BOP": 74.8,          # MUSD (replaces CAPEX!D27)
    # If tower technology: SET Solar Field Reference Cost to 125 MUSD or If PT technology: SET Solar Field Reference Cost to 115 MUSD
    "SF_aperture": 115,  # MUSD (replaces CAPEX!D28)
    # pv_modules_ref_cost_musd=0.33,# $/Wdc
    "PV_modules": 0.3,   # MUSD (replaces CAPEX!D41)
    #Fixed Trackers	0.01 ; Single Axis Tracking	0.0175
    "pv_fixed_opex_coeff": 1,
}

capex_musd, opex_musd, capex_df, opex_df = calc_capex_opex(
    excel_file, inputs, return_breakdowns=True, ref_cost_overrides=overrides
)

wacc = (debt * debt_rate * (1 - tax_rate)) + (equity * equity_rate)
lcoe = calculate_lcoe_constant(capex_musd, opex_musd, E_hybrid,lifetime_years,wacc)

sigma_capex = sum((capex_musd*10**6) / (construction_years*(1 + wacc)**t) for t in range(0, construction_years - 1))
sigma_opex = sum(1 / (1 + wacc)**t for t in range(1,lifetime_years + construction_years -1))
A = sigma_capex / sigma_opex
PPA = (A+opex_musd*10**6) / bonus['total_revenue_with_bonus']
annual_tot_revenue = PPA * bonus['total_revenue_with_bonus']

print("\n--- PV Simulation Summary ---")
for k, v in pv_summary.items():
    print(f"{k:25s}: {v:,.2f}") 

print("\n--- CSP Simulation Summary ---")
for k, v in csp_summary.items():
    print(f"{k:25s}: {v:,.2f}") 

print("\n--- TES & Heater ---")
print(f"Electric_Heater_Power_MWt : {PV_to_heater_max:,.2f}")
print(f"Tes_Capacity_MWh : {E_cap:,.2f}")
print(f"TES_Hours : {t_TES_hours:,.1f}")


print("\n--- CAPACITY FACTOR ---")
print(f"PV CSP Cap Factor : {CF_hybrid:,.2f}")
print(f"CSP Cap Factor : {CF_pb:,.2f}")


revenue = res.get("revenue_total", None)
if revenue is None:
    # compute from outputs
    price = np.array(res["price_profile"], dtype=float)           # EUR/MWh
    export = (np.array(res["pv_to_grid_MWe"], dtype=float) +
              np.array(res["pb_electric_MWe"], dtype=float))      # MWe
    dt_h = 1.0  # adjust if you solved with other dt
    revenue = float((price * export * dt_h).sum())

print("\n--- Dispatch Optimization Results ---")
print(f"Revenue (Σ α*E[t]): {revenue:,.2f} MWh")
print(f"💰 Bonus revenue (Σ α*E[t]): {bonus['bonus_revenue']:,.2f} MWh")
print(f"💰 Total revenue (with bonus) (Σ α*E[t]): {bonus['total_revenue_with_bonus']:,.2f} MWh")
print(f"✅ Firm operation hours: {bonus['firm_hours']} hours")

print("\n--- CAPEX/OPEX Input Parameters ---")
for key, val in inputs.items():
    try:
        print(f"{key:45s}: {val:,.3f}")
    except (TypeError, ValueError):
        print(f"{key:45s}: {val}")

print("\n--- CAPEX/OPEX Output Calculation ---")
print(f"Capex : {capex_musd:,.2f} MUSD")
print(f"Opex : {opex_musd:,.2f} MUSD")
print(f"Present Value Factor (S): {sigma_capex:,.2f}")
print(f"A value: {A:,.2f}")
print(f"PPA value: {PPA:,.2f} USD/MWh or {PPA/1000:,.4f} USD/kWh")
print(f"LCOE Hybrid USD: {lcoe:,.2f}")
print(f"Annual Revenue: {annual_tot_revenue/1000000:,.2f} MUSD" )

export_dispatch_to_csv(res, "output_data")




--- PV Simulation Summary ---
Annual_kWh               : 169,969,768.01
CapacityFactor_AC_%      : 22.76
NamePlate_kWdc           : 102,004.44
LandArea_m2              : 39,278,665.67

--- CSP Simulation Summary ---
NetCapacity_MWe          : 100.00
Annual_kWh               : 203,887,351.11
Tower_Height_m           : 194.23
Land_Area_acre           : 2,417.62
Solar_Field_Area_m2      : 1,529,073.51
Receiver_Design_MWt      : 728.16
CapacityFactor_%         : 27.48
SolarMultiple            : 3.00

--- TES & Heater ---
Electric_Heater_Power_MWt : 250.00
Tes_Capacity_MWh : 5,000.00
TES_Hours : 20.0

--- CAPACITY FACTOR ---
PV CSP Cap Factor : 0.68
CSP Cap Factor : 0.48

--- Dispatch Optimization Results ---
Revenue (Σ α*E[t]): 799,373.60 MWh
💰 Bonus revenue (Σ α*E[t]): 149,673.51 MWh
💰 Total revenue (with bonus) (Σ α*E[t]): 949,047.12 MWh
✅ Firm operation hours: 5438 hours

--- CAPEX/OPEX Input Parameters ---
PB Installed Capacity (Gross)                : 100,000.000
Solar Field Aperture

In [4]:
plot_week_dispatch(res, week_start_hour=4104, hours=168, save_as="viz/summer_elc.png")
plot_stack_flows(res, start_hour=4104, hours=168, save_as="viz/summer_heat.png")
plot_shares_price(res,
                                base_price=1.0,
                                bins=[0, 0.5, 1.0, np.inf],
                                bin_labels=["0.5×", "1×", "2×"],
                                dt=1.0,
                                save_as="viz/tariff_shceme_share.png")